# New to Timestral? Start here

This notebook sends one series of sales to Timestral and looks at exactly what goes to the model and what comes back. It takes about two minutes and needs no Spark.

**What Timestral is.** Timestral is a forecasting model from Mistral. It is *zero-shot*: it is not trained on your data. You pass in the history of a series, and it forecasts from that history alone. It takes numbers and returns numbers.

**What you will see.**

1. One series of history, 36 months.
2. The request sent to the model. It holds numbers and nothing else. There is not a single date in it.
3. The answer. For each future period there is a forecast, a low and a high.
4. How the future dates are made. The model does not make them. We do.

**Before you run it.** Run this in a session of the Cloudera AI project that installed u4cast. The model must be deployed: the project's **Model Deployments** page shows `timestral` as Deployed. The notebook reads two things from the session's environment: which model to call, and your own key. It never prints either.

In [ ]:
import json
import os
import urllib.request

import matplotlib.pyplot as plt
import pandas as pd

# Run from the project folder or from notebooks/: find the project folder either way.
ROOT = "." if os.path.exists("docs/sample-sales.csv") else ".."

## 1. One series of history

`docs/sample-sales.csv` is a small made-up file: units sold per month, by region and product. We take one series from it: Espresso beans in the North region.

A *series* is one thing followed over time. In the application the series come from an Iceberg table, and there can be thousands. Here one is enough.

In [ ]:
sales = pd.read_csv(f"{ROOT}/docs/sample-sales.csv", parse_dates=["month"])
series = sales[(sales["region"] == "North") & (sales["product"] == "Espresso beans")]
series = series.sort_values("month").reset_index(drop=True)

print(len(series), "months, from", series["month"].min().date(), "to", series["month"].max().date())
series.tail()

In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(series["month"], series["units"], color="#012970")
plt.title("History: units sold per month")
plt.ylim(bottom=0)
plt.show()

## 2. The request: numbers only

This is everything Timestral gets:

- `histories`: a list of series. Each series is a list of numbers, oldest first. We send one.
- `horizon`: how many future periods to forecast. We ask for 6.

Look at what is **not** there: no dates, no product name, no region, and nothing that says the data is monthly. The model sees 36 numbers in order. It works out the pattern, here a yearly cycle, from the numbers themselves.

A missing value would be sent as `null`. Timestral accepts that.

In [ ]:
HORIZON = 6

request = {
    "histories": [series["units"].astype(float).tolist()],
    "horizon": HORIZON,
}
print(json.dumps(request))

## 3. Send it

Timestral runs in this project as a *workbench model*: a Python function that Cloudera AI serves at a web address. We send the request there as JSON.

Cloudera AI wraps the request in an envelope. `accessKey` says which model to call. The `Authorization` header carries your own Cloudera AI key, because the model checks who is calling. Both come from the session's environment and are not shown here.

In [ ]:
ENDPOINT = os.environ.get("U4CAST_MODEL_ENDPOINT")
if not ENDPOINT:
    ENDPOINT = f"https://modelservice.{os.environ['CDSW_DOMAIN']}/model"


def ask_timestral(request: dict) -> dict:
    """Send one request to the workbench model and return its answer."""
    envelope = {"accessKey": os.environ["U4CAST_MODEL_ACCESS_KEY"], "request": request}
    call = urllib.request.Request(
        ENDPOINT,
        data=json.dumps(envelope).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {os.environ['CDSW_APIV2_KEY']}",
        },
        method="POST",
    )
    with urllib.request.urlopen(call, timeout=60) as reply:
        answer = json.loads(reply.read())
    if not answer.get("success"):
        raise RuntimeError(f"The model did not return a forecast: {str(answer)[:300]}")
    return answer["response"]


response = ask_timestral(request)
print(json.dumps(response, indent=1))

## 4. The answer: three numbers per future period

The answer has the same shape as the request: one row per series, and in each row one number per future period. There are three such blocks.

- `forecast` is the *median*: the model thinks the actual value is as likely to be above it as below it.
- `low` and `high` are *quantiles*. A quantile is a value the actual is expected to fall below a given share of the time. `low_level` and `high_level` say which shares: 0.1 and 0.9. So the actual should land between low and high about 8 times out of 10.
- `model_name` says which model answered, `seconds` how long it took, and `threads` how many processor threads it used.

Again there are no dates. The first number is "one period after the end of the history", the second "two periods after", and so on.

In [ ]:
print("Model:", response["model_name"], "| time:", response["seconds"], "seconds")
print("Low is the", response["low_level"], "quantile")
print("High is the", response["high_level"], "quantile")
print("Rows:", len(response["forecast"]), "| periods per row:", len(response["forecast"][0]))

## 5. The dates are ours to make

We know something the model was never told: the data is monthly, and the history ends in December 2025. So the six numbers belong to January to June 2026. We make those dates ourselves and line the numbers up with them.

This is one of the things the application does for you. It works out the frequency of a dataset, makes the future dates, and stores each forecast value with its date and its series.

In [ ]:
last_month = series["month"].max()
future_months = pd.date_range(last_month, periods=HORIZON + 1, freq="MS")[1:]

forecast = pd.DataFrame(
    {
        "month": future_months,
        "low": response["low"][0],
        "forecast": response["forecast"][0],
        "high": response["high"][0],
    }
).round(1)
assert (forecast["low"] <= forecast["forecast"]).all()
assert (forecast["forecast"] <= forecast["high"]).all()
forecast

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(series["month"], series["units"], color="#012970", label="History")
plt.plot(forecast["month"], forecast["forecast"], color="#4154f1", label="Forecast")
plt.fill_between(
    forecast["month"],
    forecast["low"],
    forecast["high"],
    color="#4154f1",
    alpha=0.15,
    label="Range, 10% to 90%",
)
plt.title("History, forecast and range")
plt.ylim(bottom=0)
plt.legend(loc="lower left")
plt.show()

## 6. Many series in one call

`histories` is a list, so one request can carry many series. Here are three products at once. The answer has three rows, in the same order.

The model forecasts each series on its own: a series gets the same forecast alone or in a batch. Sending many together is simply faster. The application sends hundreds per request.

In [ ]:
north = sales[sales["region"] == "North"]
north = north.pivot(index="month", columns="product", values="units").sort_index()

histories = [north[product].astype(float).tolist() for product in north.columns]
batch = {"histories": histories, "horizon": 3}
answer = ask_timestral(batch)

ahead = ["1 month ahead", "2 months ahead", "3 months ahead"]
pd.DataFrame(answer["forecast"], index=north.columns, columns=ahead).round(1)

## What you have seen, and what the application adds

- Timestral takes rows of numbers and a horizon. It returns a forecast, a low and a high for each future period.
- It never sees a date, a name or a frequency. Lining numbers up with dates and series is the caller's job.
- Nothing was trained. The same model forecasts any series from its history.

The u4cast application is that caller, for thousands of series. It reads history from Iceberg tables, makes the dates, sends the series in batches, and stores every forecast in Iceberg. When a forecast is made again next month, both versions are kept, so each predicted value can be traced to the forecast it came from and compared with what actually happened.

**Where to go next**

- Mistral's introduction to Timestral: https://mistral.ai/news/timestral/
- The walk through the application's screens: `docs/test-drive.md`

The Timestral weights have their own license from Mistral, separate from the license of this project's code. You are responsible for checking it and complying with it.